In [1]:
# Stockout / Overstock risk + recommended actions
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

## Load data with LightGBM predictions

In [2]:
weekly = pd.read_csv("weekly_with_lgb_predictions.csv")
weekly["week_start"] = pd.to_datetime(weekly["week_start"])
weekly = weekly.sort_values(["sku_id", "week_start"]).reset_index(drop=True)

print("Loaded:", weekly.shape)

# Use LightGBM prediction where available, else baseline
weekly["forecast"] = weekly["lgb_pred"].fillna(weekly["baseline_pred"])
weekly["forecast"] = weekly["forecast"].fillna(0).clip(lower=0)

Loaded: (52400, 28)


 # #1. Focus on the latest week (current inventory position)

In [3]:
latest_week = weekly["week_start"].max()
current = weekly[weekly["week_start"] == latest_week].copy()

print(f"Latest week: {latest_week.date()}")
print(f"SKUs in latest week: {len(current)}")

Latest week: 2026-08-10
SKUs in latest week: 200


# 2. Risk calculations

In [4]:
# Expected demand over lead time (simple: forecast * lead_time_weeks)
# lead_time is in days → convert to weeks
current["lead_time_weeks"] = (current["lead_time"] / 7).clip(lower=1)

# Demand during lead time
current["demand_during_lead"] = current["forecast"] * current["lead_time_weeks"]

# Inventory position = on_hand + on_order
current["inventory_position"] = current["on_hand"].fillna(0) + current["on_order"].fillna(0)

# Weeks of cover = inventory / weekly forecast
current["weeks_of_cover"] = np.where(
    current["forecast"] > 0,
    current["inventory_position"] / current["forecast"],
    np.inf
)

# 3. Risk flags (aligned with project decision grid)

In [5]:
# Stockout risk: inventory position < demand during lead time
current["stockout_risk"] = current["inventory_position"] < current["demand_during_lead"]

# Overstock risk: weeks of cover > 12 (example threshold — adjustable)
current["overstock_risk"] = current["weeks_of_cover"] > 12

# -------------------------------------------------
# 4. Action recommendation (4 quadrants from the brief)
# -------------------------------------------------
def recommend_action(row):
    if row["stockout_risk"] and not row["overstock_risk"]:
        return "REORDER"
    elif row["overstock_risk"] and not row["stockout_risk"]:
        return "MARKDOWN / REDUCE"
    elif row["stockout_risk"] and row["overstock_risk"]:
        return "WATCH"          # unusual — check data
    else:
        return "HEALTHY"

current["action"] = current.apply(recommend_action, axis=1)

# 5. Approximate ₹ impact

In [6]:
# Sales at risk (stockout) ≈ forecast * avg_price for at-risk SKUs
current["sales_at_risk"] = np.where(
    current["stockout_risk"],
    current["forecast"] * current["avg_price"],
    0
)

# Locked capital (overstock) ≈ excess units * unit cost proxy
# excess ≈ inventory beyond 8 weeks of cover
current["excess_units"] = np.where(
    current["weeks_of_cover"] > 8,
    current["inventory_position"] - (8 * current["forecast"]),
    0
)
current["locked_capital"] = current["excess_units"] * current["avg_price"] * 0.6  # rough cost proxy

# 6. Summary

In [7]:
print("\n" + "="*55)
print("RISK SCORING SUMMARY (Latest Week)")
print("="*55)

print("\nAction counts:")
print(current["action"].value_counts().to_string())

print(f"\nSKUs with Stockout Risk : {current['stockout_risk'].sum()}")
print(f"SKUs with Overstock Risk: {current['overstock_risk'].sum()}")

print(f"\nTotal Sales at Risk (₹) : {current['sales_at_risk'].sum():,.0f}")
print(f"Total Locked Capital (₹): {current['locked_capital'].sum():,.0f}")

print("\n--- Top 10 REORDER (highest sales at risk) ---")
reorder = current[current["action"] == "REORDER"].nlargest(10, "sales_at_risk")
print(reorder[["sku_id", "category", "forecast", "inventory_position", "weeks_of_cover", "sales_at_risk"]].to_string(index=False))

print("\n--- Top 10 MARKDOWN / REDUCE (highest locked capital) ---")
markdown = current[current["action"] == "MARKDOWN / REDUCE"].nlargest(10, "locked_capital")
print(markdown[["sku_id", "category", "forecast", "inventory_position", "weeks_of_cover", "locked_capital"]].to_string(index=False))

# -------------------------------------------------


RISK SCORING SUMMARY (Latest Week)

Action counts:
action
HEALTHY              193
REORDER                5
MARKDOWN / REDUCE      2

SKUs with Stockout Risk : 5
SKUs with Overstock Risk: 2

Total Sales at Risk (₹) : 99,745
Total Locked Capital (₹): 197,699

--- Top 10 REORDER (highest sales at risk) ---
 sku_id         category   forecast  inventory_position  weeks_of_cover  sales_at_risk
SKU0144        Furniture  53.083452                70.0        1.318678   46078.028469
SKU0143 Small Appliances 112.975613               239.0        2.115501   42260.787490
SKU0199            Decor  81.359425                65.0        0.798924    5990.494469
SKU0073 Small Appliances  36.544542               109.0        2.982662    2932.334089
SKU0048 Small Appliances  60.408929               172.0        2.847261    2483.411072

--- Top 10 MARKDOWN / REDUCE (highest locked capital) ---
 sku_id  category  forecast  inventory_position  weeks_of_cover  locked_capital
SKU0190 Furniture 54.961399     

# 7. Save risk table

In [8]:
risk_cols = [
    "sku_id", "category", "subcategory", "week_start",
    "forecast", "on_hand", "on_order", "inventory_position",
    "lead_time", "weeks_of_cover", "stockout_risk", "overstock_risk",
    "action", "sales_at_risk", "locked_capital"
]
current[risk_cols].to_csv("sku_risk_scores.csv", index=False)
print("\nSaved: sku_risk_scores.csv")
print("Stage 8 (Risk Scoring) complete.")


Saved: sku_risk_scores.csv
Stage 8 (Risk Scoring) complete.
